# 🎯 InklusiKerja — Step 3: Recommendation Engine (v4)

**Prerequisite:** Jalankan `01_preprocessing.ipynb` dan `02_embedding.ipynb` terlebih dahulu.

**Yang dilakukan:**
- Load FAISS index + metadata
- Encode query kandidat
- Hard filter disability + semantic threshold
- Fallback jika hasil kurang dari top_k
- Scoring: `disability=60%, semantic=35%, skill=5%`

## 1. Import & Konstanta

In [ ]:
import os, re, json, pickle
import numpy as np
import pandas as pd
from typing import Optional
from dataclasses import dataclass, field, asdict

DISABILITY_MATCH_THRESHOLD = 0.5
INDEX_DIR = "data/index"

SKILL_ALIAS = {
    "r": "r programming", "go": "golang",
    "machine learning": "machine learning dasar", "ml": "machine learning dasar",
    "node": "node.js", "nodejs": "node.js", "vue": "vue.js",
    "postgres": "postgresql", "ms excel": "microsoft excel",
    "ms word": "microsoft word", "ms office": "microsoft office",
    "power point": "microsoft powerpoint", "google sheet": "google sheets",
}

KNOWN_SKILLS = {
    "python", "r programming", "golang", "java", "javascript", "typescript",
    "php", "kotlin", "swift", "dart", "css", "html",
    "react", "vue.js", "node.js", "django", "fastapi", "spring boot",
    "laravel", "flutter", "react native", "bootstrap", "jetpack compose",
    "redis", "microservices", "rest api", "git",
    "power bi", "tableau", "google data studio", "numpy", "pandas",
    "sql", "mysql", "postgresql", "sqlite", "excel", "google sheets",
    "machine learning dasar", "statistik", "data cleaning", "scikit-learn",
    "microsoft excel", "docker", "kubernetes", "aws", "gcp", "azure",
    "linux", "ci/cd", "jenkins", "terraform", "ansible",
    "figma", "adobe xd", "sketch", "invision", "canva",
    "adobe illustrator", "adobe photoshop", "coreldraw",
    "wireframing", "prototyping", "user research", "design system",
    "seo", "google ads", "facebook ads", "tiktok ads",
    "email marketing", "copywriting", "content creation",
    "digital marketing", "analitik media sosial",
    "akuntansi", "jurnal keuangan", "pajak", "rekonsiliasi bank",
    "laporan keuangan", "audit internal", "sap", "myob", "accurate", "erp",
    "hris", "rekrutmen", "payroll", "manajemen kinerja",
    "pelatihan sdm", "hubungan industrial", "linkedin recruiter",
    "firewall", "owasp", "penetration testing", "metasploit",
    "ethical hacking", "kriptografi", "iso 27001", "siem", "nmap",
    "selenium", "cypress", "katalon", "postman", "jira",
    "qa documentation", "test case", "testing manual", "bug tracking",
    "cisco", "mikrotik", "vpn", "ccna",
    "monitoring jaringan", "server management", "networking",
    "microsoft office", "microsoft word", "google workspace",
    "administrasi perkantoran", "zendesk", "freshdesk", "crm", "live chat",
    "komunikasi", "problem solving", "empati", "penanganan keluhan",
    "notulensi", "manajemen jadwal", "pengarsipan",
    "wms", "sap wm", "inventory", "supply chain", "manajemen gudang",
    "perencanaan distribusi", "e-learning", "zoom", "google classroom",
    "moodle", "microsoft teams", "kurikulum",
    "motion graphics", "tipografi", "desain visual", "branding",
    "editing video", "capcut", "youtube", "instagram",
    "penulisan konten", "seo writing", "wordpress",
    "ketelitian data", "pengetikan cepat", "spreadsheet",
    "penerjemahan", "proofreading", "lokalisasi", "sdl trados",
    "bahasa inggris", "bahasa jepang", "bahasa mandarin",
}

print("✅ Import & konstanta siap")

## 2. Data Classes & Helper Functions

In [ ]:
def normalize_skill_name(s: str) -> str:
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return SKILL_ALIAS.get(s, s)

def extract_skills_from_text(text: str) -> set:
    text_lower = text.lower()
    for alias, canonical in SKILL_ALIAS.items():
        text_lower = re.sub(r"\b" + re.escape(alias) + r"\b", canonical, text_lower)
    found = set()
    for skill in KNOWN_SKILLS:
        if re.search(r"\b" + re.escape(skill) + r"\b", text_lower):
            found.add(skill)
    return found

def compute_skill_gap(kandidat_skills, qualification_text, skill_tags=None):
    kandidat_set = {normalize_skill_name(s) for s in kandidat_skills}
    required_set = set(skill_tags) if skill_tags else extract_skills_from_text(qualification_text)
    if not required_set:
        return list(kandidat_skills[:3]), [], 0.4
    matched = kandidat_set & required_set
    missing = required_set - kandidat_set
    return list(matched), list(missing), len(matched) / len(required_set)

@dataclass
class KandidatProfile:
    disability_type: str
    skills: list
    functional_profile: str
    preferred_level: Optional[str] = None
    location: Optional[str] = None

    def to_query_text(self) -> str:
        normalized = [normalize_skill_name(s) for s in self.skills]
        query = (
            f"Kandidat dengan {self.disability_type} mencari pekerjaan di bidang "
            f"{', '.join(normalized[:3])}. "
            f"Memiliki skill: {', '.join(normalized)}. "
            f"{self.functional_profile}"
        )
        if self.preferred_level:
            query += f" Level yang diinginkan: {self.preferred_level}."
        return query

    def normalized_skills(self):
        return [normalize_skill_name(s) for s in self.skills]

@dataclass
class JobRecommendation:
    rank: int
    job_id: str
    job_title: str
    disability_type: str
    level: str
    qualification: str
    accessibility: str
    semantic_score: float
    skill_match_score: float
    disability_match_score: float
    final_score: float
    skill_gap: list
    matched_skills: list
    accommodation_suggestions: list
    explanation: str

print("✅ Data classes & helpers siap")

## 3. Accommodation Map

In [ ]:
ACCOMMODATION_MAP = {
    "tunanetra": [
        "Screen reader (NVDA/JAWS/VoiceOver)", "Display braille atau braille note taker",
        "Komputer dengan antarmuka berbasis keyboard shortcut",
        "Dokumen dan materi kerja dalam format aksesibel (PDF tertagged)",
        "Workstation dengan monitor kontras tinggi atau konfigurasi audio",
    ],
    "tunarungu": [
        "Interpreter BISINDO/SIBI untuk meeting penting",
        "Captioning real-time (CART) untuk rapat dan presentasi",
        "Komunikasi via teks: Slack, email, atau aplikasi chat",
        "Visual alert system (lampu kedip) untuk alarm/notifikasi",
    ],
    "tunawicara": [
        "Perangkat augmentative & alternative communication (AAC)",
        "Text-to-speech software untuk presentasi",
        "Preferensi komunikasi tertulis/digital untuk semua koordinasi",
    ],
    "gangguan daksa tangan": [
        "Voice recognition software (Dragon NaturallySpeaking)",
        "Mouse trackball atau joystick ergonomis",
        "Keyboard one-hand atau keyboard adaptif",
    ],
    "gangguan daksa kaki": [
        "Aksesibilitas fisik: ramp, lift, pintu otomatis",
        "Parkir prioritas dekat pintu masuk",
        "Toilet aksesibel kursi roda",
    ],
    "autisme": [
        "Lingkungan kerja terstruktur dengan jadwal konsisten",
        "Ruang kerja tenang atau noise-cancelling headphone",
        "Instruksi tugas tertulis yang jelas dan terperinci",
    ],
    "gangguan spektrum autisme": [
        "Lingkungan kerja terstruktur dengan jadwal konsisten",
        "Ruang kerja tenang atau noise-cancelling headphone",
    ],
    "gangguan mental": [
        "Jadwal kerja yang konsisten dan dapat diprediksi",
        "Akses ke layanan konseling atau EAP",
        "Beban kerja yang realistis tanpa overtime berlebihan",
    ],
    "gangguan intelektual": [
        "Instruksi kerja sederhana dengan visual aids",
        "Mentor/buddy system di tempat kerja",
    ],
}

def get_accommodation(disability_type: str) -> list:
    key = disability_type.lower().strip()
    if key in ACCOMMODATION_MAP:
        return ACCOMMODATION_MAP[key]
    for k, v in ACCOMMODATION_MAP.items():
        if k in key or key in k:
            return v
    return [
        "Konsultasikan kebutuhan akomodasi spesifik dengan HR",
        "Evaluasi ergonomis workstation",
        "Fleksibilitas jadwal kerja sesuai kebutuhan",
    ]

print("✅ Accommodation map siap")

## 4. Recommendation Engine Class

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

class RecommendationEngine:

    def __init__(self, index_dir=INDEX_DIR, model_key="multilingual_minilm"):
        print("🚀 Memuat RecommendationEngine v4...")

        with open(f"{index_dir}/config.json") as f:
            self.config = json.load(f)

        self.index = faiss.read_index(f"{index_dir}/jobs.faiss")
        print(f"   ✓ FAISS index: {self.index.ntotal} vektor")

        with open(f"{index_dir}/jobs_metadata.pkl", "rb") as f:
            self.metadata = pickle.load(f)
        print(f"   ✓ Metadata: {len(self.metadata)} entri")

        model_map = {
            "multilingual_minilm": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            "finetuned": "models/finetuned",
        }
        model_name = model_map.get(model_key, model_map["multilingual_minilm"])
        self.model = SentenceTransformer(model_name)
        print(f"   ✓ Model: {model_name}")
        print("✅ Engine siap!")

    def _encode_query(self, text: str):
        vec = self.model.encode([text], normalize_embeddings=True, convert_to_numpy=True)
        return vec.astype(np.float32)

    def _compute_disability_match(self, kandidat_disability, job_disability):
        k = kandidat_disability.lower().strip()
        j = job_disability.lower().strip()
        if k == j: return 1.0
        if k in j or j in k: return 0.8
        groups = [
            {"tunanetra", "gangguan penglihatan"},
            {"tunarungu", "gangguan pendengaran", "tuli"},
            {"tunawicara", "gangguan bicara"},
            {"tunadaksa", "gangguan daksa", "gangguan daksa kaki", "gangguan daksa tangan"},
            {"autisme", "gangguan spektrum autisme", "asd"},
            {"gangguan mental", "psikososial"},
            {"disabilitas kognitif", "gangguan intelektual"},
        ]
        for g in groups:
            if any(key in k for key in g) and any(key in j for key in g):
                return 0.9
        return 0.1

    def _score_candidate(self, faiss_idx, sem_score, kandidat, weights):
        meta = self.metadata.get(int(faiss_idx), {})
        if not meta: return None

        skill_tags = meta.get("skill_tags", [])
        if isinstance(skill_tags, str):
            try: import ast; skill_tags = ast.literal_eval(skill_tags)
            except: skill_tags = []

        matched_skills, skill_gap, skill_ratio = compute_skill_gap(
            kandidat.normalized_skills(), meta.get("qualification", ""), skill_tags=skill_tags)
        dis_score = self._compute_disability_match(kandidat.disability_type, meta.get("disability_type", ""))
        sem_norm  = float(np.clip(sem_score, 0, 1))

        level_bonus = 0.0
        if kandidat.preferred_level:
            if kandidat.preferred_level.lower() in meta.get("level", "").lower():
                level_bonus = 0.05

        final = (weights["semantic"] * sem_norm + weights["skill"] * skill_ratio +
                 weights["disability"] * dis_score + level_bonus) * 100

        explanation_parts = []
        if sem_norm > 0.8: explanation_parts.append("Sangat relevan secara semantik dengan profil kamu.")
        elif sem_norm > 0.6: explanation_parts.append("Cukup relevan dengan profil dan pengalamanmu.")
        elif sem_norm > 0.4: explanation_parts.append("Ada kemiripan pada beberapa aspek profil.")
        else: explanation_parts.append("Relevansi semantik rendah — pertimbangkan opsi lain.")
        if matched_skills: explanation_parts.append(f"Skill yang sudah kamu miliki: {', '.join(matched_skills[:3])}.")
        if skill_gap: explanation_parts.append(f"Perlu diperkuat: {', '.join(skill_gap[:3])}.")
        if dis_score >= 0.9: explanation_parts.append("Posisi ini dirancang khusus untuk profil disabilitasmu.")
        elif dis_score < 0.5: explanation_parts.append("⚠️ Profil disabilitas tidak sesuai posisi ini (fallback).")

        return JobRecommendation(
            rank=0, job_id=meta["job_id"], job_title=meta["job_title"],
            disability_type=meta["disability_type"], level=meta["level"],
            qualification=meta["qualification"], accessibility=meta["accessibility"],
            semantic_score=round(sem_norm * 100, 1),
            skill_match_score=round(skill_ratio * 100, 1),
            disability_match_score=round(dis_score * 100, 1),
            final_score=round(final, 1),
            skill_gap=skill_gap[:5], matched_skills=matched_skills[:5],
            accommodation_suggestions=get_accommodation(kandidat.disability_type),
            explanation=" ".join(explanation_parts),
        )

    def recommend(self, kandidat, top_k=10, weights=None, filter_disability=True,
                  disability_threshold=DISABILITY_MATCH_THRESHOLD, min_semantic=0.43):
        if weights is None:
            weights = {"semantic": 0.35, "skill": 0.05, "disability": 0.60}
        query_vec = self._encode_query(kandidat.to_query_text())
        n_retrieve = self.index.ntotal if filter_disability else min(top_k * 10, self.index.ntotal)
        scores, indices = self.index.search(query_vec, n_retrieve)
        matched, fallback = [], []
        for faiss_idx, sem_score in zip(indices[0], scores[0]):
            if faiss_idx == -1: continue
            rec = self._score_candidate(faiss_idx, sem_score, kandidat, weights)
            if rec is None: continue
            passes_d = (not filter_disability) or (rec.disability_match_score / 100 >= disability_threshold)
            passes_s = rec.semantic_score / 100 >= min_semantic
            if passes_d and passes_s: matched.append(rec)
            else: fallback.append(rec)
        matched.sort(key=lambda r: r.final_score, reverse=True)
        if len(matched) < top_k:
            fallback.sort(key=lambda r: r.semantic_score * 0.70 + r.skill_match_score * 0.30, reverse=True)
            shortage = top_k - len(matched)
            matched.extend(fallback[:shortage])
            if shortage > 0: print(f"   ⚠️  Menambahkan {shortage} job fallback.")
        final_results = matched[:top_k]
        for i, r in enumerate(final_results, start=1): r.rank = i
        return final_results

print("✅ RecommendationEngine class siap")

## 5. Inisialisasi Engine

In [ ]:
# Pilih model_key: 'multilingual_minilm' atau 'finetuned'
engine = RecommendationEngine(index_dir=INDEX_DIR, model_key="multilingual_minilm")

## 6. Demo — Uji Rekomendasi

In [ ]:
kandidat = KandidatProfile(
    disability_type="Gangguan Penglihatan (Tunanetra)",
    skills=["R", "Power BI", "NumPy", "Tableau", "Machine Learning Dasar"],
    functional_profile=(
        "Mengalami gangguan penglihatan total sejak lahir. "
        "Ahli data analyst dan business intelligence menggunakan JAWS. "
        "Lulus S1 Teknik Informatika."
    ),
    preferred_level="Mid level",
)

print(f"📝 Query: {kandidat.to_query_text()}\n")

In [ ]:
results = engine.recommend(kandidat, top_k=5)

print("=" * 60)
print("TOP REKOMENDASI")
print("=" * 60)
for r in results:
    print(f"\n#{r.rank} [{r.final_score:.1f}] {r.job_title} — {r.level}")
    print(f"  Disability: {r.disability_type}")
    print(f"  Scores → Semantic:{r.semantic_score:.1f}% | Skill:{r.skill_match_score:.1f}% | Disability:{r.disability_match_score:.1f}%")
    print(f"  Matched Skills : {r.matched_skills}")
    print(f"  Skill Gap      : {r.skill_gap}")
    print(f"  💬 {r.explanation}")

## 7. Simpan Hasil ke JSON

In [ ]:
import os
os.makedirs("data/processed", exist_ok=True)
out_path = "data/processed/sample_recommendations.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([asdict(r) for r in results], f, ensure_ascii=False, indent=2)
print(f"💾 Tersimpan → {out_path}")
print("\n➡️  Lanjut ke 04_api.ipynb atau 05_evaluation.ipynb")